# Zurich Repeat Customer Analysis
## Exploratory Data Analysis: Historical Data for New Business Decisions

**Objective**: Analyze repeat customer patterns, product trends, broker dynamics, and decision outcomes to inform risk scoring and recommendations.

**Data Period**: 2021-2026 (46K+ submissions)

## 1. Setup Environment

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('[SETUP] Environment configured')

[SETUP] Environment configured


## 2. Import & Load Data

In [ ]:
# Navigate to project root and load processed CSVs
import os
os.chdir('..')  # Move to c:\PROJECT\

repeat_customers = pd.read_csv('data/parsed/repeat_customers.csv')
all_submissions = pd.read_csv('data/parsed/all_submissions.csv')
customer_broker = pd.read_csv('data/parsed/customer_broker_relationships.csv')
product_trends = pd.read_csv('data/parsed/product_trends.csv')
status_product = pd.read_csv('data/parsed/status_by_product.csv')
companies_pdfs = pd.read_csv('data/parsed/companies_with_pdfs.csv')

print(f'[LOAD] Repeat customers: {len(repeat_customers)}')
print(f'[LOAD] All submissions: {len(all_submissions)}')
print(f'[LOAD] Customer-Broker pairs: {len(customer_broker)}')
print(f'[LOAD] Companies with PDFs: {len(companies_pdfs)}')

FileNotFoundError: [Errno 2] No such file or directory: 'data/parsed/repeat_customers.csv'

## 3. Repeat Customer Analytics

In [ ]:
# Distribution of repeat customer submissions
print('=== REPEAT CUSTOMER DISTRIBUTION ===')
print(repeat_customers['submission_count'].describe())
print(f"\nCustomers with >5 submissions: {(repeat_customers['submission_count'] > 5).sum()}")
print(f"Customers with >10 submissions: {(repeat_customers['submission_count'] > 10).sum()}")

# Visualize
fig = px.histogram(repeat_customers, x='submission_count', nbins=30, title='Distribution of Repeat Customer Submissions')
fig.update_xaxes(title='Number of Submissions')
fig.update_yaxes(title='Number of Customers')
fig.show()

# Top repeat customers
print(f'\n=== TOP 10 REPEAT CUSTOMERS ===')
print(repeat_customers.head(10).to_string())

## 4. Product Analysis

In [ ]:
# Product trends over time
print('=== TOP PRODUCTS BY SUBMISSION COUNT ===')
product_summary = product_trends.groupby('Product Name')['count'].sum().sort_values(ascending=False)
print(product_summary.head(10))

# Visualize product trends
top_products = product_summary.head(5).index.tolist()
df_top_products = product_trends[product_trends['Product Name'].isin(top_products)]

fig = px.line(df_top_products, x='Requested Coverage Effective Year', y='count', color='Product Name',
              title='Top 5 Products Over Time')
fig.update_yaxes(title='Submission Count')
fig.show()

## 5. Broker Intelligence

In [ ]:
# Top brokers by customer count
broker_stats = customer_broker.groupby('National Broker Name').agg({
    'count': ['sum', 'count']
}).reset_index()
broker_stats.columns = ['Broker', 'Total Submissions', 'Unique Customers']
broker_stats = broker_stats.sort_values('Total Submissions', ascending=False)

print('=== TOP 10 BROKERS ===')
print(broker_stats.head(10).to_string())

# Visualize
fig = px.bar(broker_stats.head(10), x='Broker', y='Total Submissions', 
             title='Top 10 Brokers by Submission Count')
fig.update_xaxes(tickangle=-45)
fig.show()

## 6. Submission Status Analysis

In [ ]:
# Status distribution
status_summary = status_product.groupby('Current Status Description')['count'].sum().sort_values(ascending=False)

print('=== SUBMISSION STATUS DISTRIBUTION ===')
print(status_summary)
print(f"\nApproval rate (Bound/Total): {status_summary.get('Bound', 0) / status_summary.sum():.1%}")
print(f"Decline rate: {status_summary.get('Declined', 0) / status_summary.sum():.1%}")

# Visualize
fig = px.pie(status_product.groupby('Current Status Description')['count'].sum().reset_index(),
             names='Current Status Description', values='count',
             title='Submission Status Distribution')
fig.show()

## 7. Product-Status Relationship

In [ ]:
# Approval rate by product
product_status_pivot = status_product.pivot_table(
    index='Product Name',
    columns='Current Status Description',
    values='count',
    fill_value=0
)

if 'Bound' in product_status_pivot.columns:
    product_status_pivot['Approval Rate'] = product_status_pivot['Bound'] / product_status_pivot.sum(axis=1)
    product_status_pivot = product_status_pivot.sort_values('Approval Rate', ascending=False)
    
    print('=== APPROVAL RATE BY PRODUCT (TOP 10) ===')
    print(product_status_pivot[['Approval Rate']].head(10))
    
    # Visualize
    top_products_by_volume = status_product.groupby('Product Name')['count'].sum().nlargest(10).index
    df_viz = product_status_pivot.loc[top_products_by_volume][['Approval Rate']].reset_index()
    
    fig = px.bar(df_viz, x='Product Name', y='Approval Rate',
                title='Approval Rate by Top 10 Products')
    fig.update_yaxes(tickformat='.0%')
    fig.update_xaxes(tickangle=-45)
    fig.show()

## 8. Companies with PDF Data

In [ ]:
# Summary of companies with PDF submissions available
print('=== COMPANIES WITH PDF SUBMISSIONS ===')
pdf_summary = companies_pdfs.groupby('Samples Provided').agg({
    'Submission Account Name': 'count',
    'Product Name': lambda x: x.value_counts().index[0] if len(x) > 0 else 'N/A'
}).reset_index()
pdf_summary.columns = ['Company', 'Submission Count', 'Primary Product']
pdf_summary = pdf_summary.sort_values('Submission Count', ascending=False)
print(pdf_summary.to_string())

print(f'\nTotal companies with PDFs: {len(pdf_summary)}')
print(f'Total submissions with PDFs: {len(companies_pdfs)}')

## 9. Key Insights Summary

In [ ]:
print('=== KEY FINDINGS FOR RECOMMENDATION ENGINE ===')
print()
print('1. REPEAT CUSTOMER PREVALENCE')
print(f'   - {len(repeat_customers):,} unique customers with repeat submissions')
print(f'   - Average repeats per customer: {repeat_customers["submission_count"].mean():.1f}')
print(f'   - Max repeats: {repeat_customers["submission_count"].max()} submissions')
print()
print('2. PRODUCT CONCENTRATION')
top_prod = product_summary.head(3)
for i, (prod, count) in enumerate(top_prod.items(), 1):
    pct = 100 * count / product_summary.sum()
    print(f'   {i}. {prod}: {count:,} ({pct:.1f}%)')
print()
print('3. BROKER CONCENTRATION')
top_brokers = broker_stats.head(3)
for i, row in top_brokers.iterrows():
    print(f'   {i+1}. {row["Broker"]}: {int(row["Total Submissions"]):,} submissions')
print()
print('4. OUTCOME DISTRIBUTION')
print(f'   - Bound (Approved): {status_summary.get("Bound", 0):,} ({100*status_summary.get("Bound", 0)/status_summary.sum():.1f}%)')
print(f'   - Declined: {status_summary.get("Declined", 0):,} ({100*status_summary.get("Declined", 0)/status_summary.sum():.1f}%)')
print(f'   - Quote Not Taken: {status_summary.get("Quote not taken", 0):,} ({100*status_summary.get("Quote not taken", 0)/status_summary.sum():.1f}%)')

## 10. Export Summary for Dashboard

In [ ]:
# Create summary stats for dashboard initialization
summary_stats = {
    'total_repeat_customers': len(repeat_customers),
    'total_submissions': len(all_submissions),
    'avg_submissions_per_customer': repeat_customers['submission_count'].mean(),
    'top_broker': broker_stats.iloc[0]['Broker'],
    'approval_rate': status_summary.get('Bound', 0) / status_summary.sum(),
    'decline_rate': status_summary.get('Declined', 0) / status_summary.sum(),
    'companies_with_pdfs': len(pdf_summary),
}

print('\n=== SUMMARY STATS FOR DASHBOARD ===')
for key, val in summary_stats.items():
    if isinstance(val, float):
        print(f'{key}: {val:.2%}' if key.endswith('_rate') else f'{key}: {val:.2f}')
    else:
        print(f'{key}: {val}')

# Save for reference
pd.DataFrame([summary_stats]).to_csv('outputs/dashboard_summary_stats.csv', index=False)
print('\n[EXPORT] Summary stats saved to outputs/dashboard_summary_stats.csv')